In [ ]:
# 0. 패키지 버전 맞추기 — Colab에서 처음 한 번: 이 셀 실행 → 런타임 다시 시작 → Run All
import sys
if 'google.colab' in sys.modules:
    !pip install -q pandas==3.1.0 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 pyarrow==26.0.0 PyYAML==6.0.3
    print('설치 끝: 처음이면 런타임 → 세션 다시 시작 후 Run All')
else:
    print('로컬: 설치 생략')

# 튜닝 4-2 — XGBoost 순위 회귀·pairwise: 깊이 3·4 × 트리 상한 2000 × L2 0·5 (Colab GPU)

- **조건(모든 변형 같음)**: 주 세트(기본+H1+H3+H2+H4, RMW 포함), 학습률 0.01, 16 Fold, 내부 검증 직전 1년, H3 λ 3개. 전처리는 basefeat 실행과 같다. pairwise는 표본 8개 쌍(seed 0, 2026-11-22 확인 결과).
- **트리**: 2000개까지 한 번 학습하고 해마다 선택(1~2000)과 500·1000·1500·2000 고정을 같은 학습에서 잘라 비교.
- **판단**: 같은 트리 기준끼리 깊이 3 vs 4, L2 0 vs 5. 0.002 미만이면 단순한 쪽(깊이 3, L2 0, 트리 적은 쪽).
- **중간 저장**: 단위(모델 × 깊이 × L2 = 8개)마다 `model_results/<AUTHOR>/tune_xgb_v1_parts/`에 저장, 끊기면 Run All로 이어감. 요약 표는 `model_results/<AUTHOR>/tune_xgb_v1_summary.csv`. 예상 약 2시간.

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
else:
    raise RuntimeError('GPU 런타임이 아니다(XGBoost는 cuda로 돌린다)')

## 1. 설정 (Drive 연결·코드 받기)

In [ ]:
import os, sys, subprocess
from datetime import datetime
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JiunShin1214/finance_analytics.git'
GIT_REF = 'jiun'                # 브랜치 이름 — 그 브랜치의 GitHub 최신 코드
AUTHOR = 'jiun'                 # 결과 폴더 model_results/<AUTHOR>/ (영문·숫자·_·-만)
RUN_NAME = 'tune_xgb_v1'        # 같은 이름으로 다시 Run All 하면 Drive에 저장된 단위를 이어 씀
DRIVE_BASE = Path('/content/drive/MyDrive/금융데이터인텔리전스')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not GIT_REF:
        raise RuntimeError("GIT_REF에 브랜치 이름을 적는다.")
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', '--prune'], check=True)
    _remote = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--verify', '--quiet', f'origin/{GIT_REF}'],
                             capture_output=True, text=True).stdout.strip()
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', _remote or GIT_REF], check=True)
    print('받은 코드:', GIT_REF, '@', subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    if not (ROOT / 'data').is_symlink():       # 저장소 data/를 Drive 공유 data로 연결
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(DRIVE_BASE / 'data')
    OUT_ROOT = Path('/content/drive/MyDrive/model_results')   # 중간 저장(parts)과 결과 모두 내 드라이브 → 세션이 끊겨도 남음
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_ROOT = None

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
from src.evaluation import model_test as mt
from src.evaluation import base_model_runs as bmr
from src.features.h3 import compute_h3_sigma_grid

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
mt.check_no_test(config)
need = [config['PATHS'][k] for k in ('panel', 'eligibility', 'h1_components', 'base_features', 'h2', 'h4')]
missing = [n for n in need if not (ROOT / n).is_file()]
if missing:
    raise FileNotFoundError('Drive data/에 없음: ' + ', '.join(missing))
STARTED = datetime.now()
import numpy as np
MAX_TREES = 2000                # 트리 상한(기존 1000). 2000까지 한 번 학습하고 아래 고정 개수로 잘라 비교
FIXED_TREES = [500, 1000, 1500, 2000]
SET = 'base+h1+h3+h2+h4'        # 주 세트(RMW 포함)
mt.boosting.MAX_TREES = MAX_TREES   # 노트북 안에서만 상한을 바꾼다(코드 기본값 1000은 그대로)
PARTS = bmr.parts_dir(ROOT, config, author=AUTHOR, run_name=RUN_NAME, out_root=OUT_ROOT)
print('중간 저장 폴더:', PARTS)
print('이미 저장된 단위:', len(list(PARTS.glob('*.pkl'))))

## 2. H3 격자·전처리

In [ ]:
panel = pd.read_parquet(ROOT / config['PATHS']['panel'], columns=['stock_id', 'month', 'return', 'eligible'])
eligibility = pd.read_parquet(ROOT / config['PATHS']['eligibility'])
H3_GRID = compute_h3_sigma_grid(panel, eligibility)
del panel, eligibility
PREP = bmr.prepare(ROOT, config, 'boosting', input_transform='rank', h3_grid=H3_GRID)
COLS = bmr.input_sets('boosting')[SET]
assert 'rmw' in COLS, 'RMW가 없는 코드·피처 표다'

## 3. 실행 (8단위)

In [ ]:
PAIR_MEAN8 = {'lambdarank_pair_method': 'mean', 'lambdarank_num_pair_per_sample': 8, 'seed': 0}   # 2026-11-22 확인: 전체 쌍 대신 표본 8개
KINDS = {'xgb_rankreg': ('percentile', 'mse'), 'xgb_pairwise': ('decile', 'pairwise')}
fp = bmr.fingerprint(ROOT, config, 'boosting', 'rank')
units = []
for kind, (target, loss) in KINDS.items():
    for depth in (3, 4):
        for l2 in (0, 5):
            ov = {**(PAIR_MEAN8 if kind == 'xgb_pairwise' else {}), **({'lambda': l2} if l2 else {})} or None
            setting = [(kind, target, loss, depth, 0.01)]
            spec = {'set': SET, 'setting': setting, 'overrides': ov, 'max_trees': MAX_TREES, 'fixed_trees': FIXED_TREES}
            units.append((f'tune_xgb__{kind}__d{depth}__l2{l2}', spec,
                          (lambda kind=kind, setting=setting, ov=ov: mt.run_boosting(PREP, config, kind, phase='basefeat', settings=setting,
                                                                                    sets={SET: COLS}, overrides=ov, fixed_trees=FIXED_TREES))))
RESULTS = bmr.run_units(units, PARTS, fp)

## 4. 요약

In [ ]:
rows = []
for unit, r in RESULTS.items():
    _, kind, d, l2 = unit.split('__')
    vk = r['val_keys']
    sel = r['selections']
    for key, pred in r['predictions'].items():
        hp = key.split('|')[-1]
        basis = '해마다' if '_t' not in hp.split('seed0')[-1] else hp.split('_t')[-1] + ' 고정'
        ic = mt.monthly_ic(vk.target_month.to_numpy(), pred, vk.y.to_numpy()).ic
        s = sel[sel.hparams == hp]
        rows.append({'모델': kind, '깊이': int(d[1:]), 'L2': int(l2[2:]), '트리': basis, '평균 IC': ic.mean(), 'ICIR': ic.mean() / ic.std(ddof=1),
                     '학습 IC': s.train_ic.mean(), '학습−검증': s.train_ic.mean() - ic.mean(), '고른 트리 중앙값': s.trees.median(),
                     f'상한 근처(≥{MAX_TREES - 10})': int((s.trees >= MAX_TREES - 10).sum()), '개월': int(ic.notna().sum()),
                     '학습 시간(분)': r['timings'].seconds.sum() / 60})
T = pd.DataFrame(rows)
T['트리'] = pd.Categorical(T['트리'], ['해마다'] + [f'{n} 고정' for n in FIXED_TREES], ordered=True)
T = T.sort_values(['모델', '깊이', 'L2', '트리']).reset_index(drop=True)
print(T.round(4).to_string())
out = PARTS.parent / f'{RUN_NAME}_summary.csv'
T.to_csv(out, index=False, encoding='utf-8-sig')
print('저장:', out)